Fix: Recovery Cell

In [3]:
# ============================================
# RECOVERY CELL: Restore Everything After Session Restart
# ============================================

# ============ 1. MOUNT DRIVE ============
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from torchvision import transforms
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ============ 2. SET PATHS ============
PROJECT_DIR = '/content/drive/MyDrive/AccessoriesAI'
MODELS_DIR = os.path.join(PROJECT_DIR, 'trained_models')
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
PLOTS_DIR = os.path.join(PROJECT_DIR, 'plots')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ============ 3. LOAD SAVED ENCODERS ============
with open(os.path.join(MODELS_DIR, 'dress_label_encoders.json'), 'r') as f:
    encoder_data = json.load(f)

# Rebuild encoders
encoders = {}
num_classes = {}
for attr_name, attr_info in encoder_data.items():
    enc = LabelEncoder()
    enc.classes_ = np.array(attr_info['classes'])
    encoders[attr_name] = enc
    num_classes[attr_name] = attr_info['num_classes']

print(f"\n✅ Encoders restored:")
for k, v in num_classes.items():
    print(f"   {k}: {v} classes")

# ============ 4. REBUILD MODEL ARCHITECTURE ============
class DressAttributeExtractor(nn.Module):
    def __init__(self, num_classes_dict):
        super(DressAttributeExtractor, self).__init__()
        self.backbone = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V2)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.shared_fc = nn.Sequential(
            nn.Linear(num_features, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
        )

        self.heads = nn.ModuleDict()
        for attr_name, n_classes in num_classes_dict.items():
            if n_classes >= 10:
                self.heads[attr_name] = nn.Sequential(
                    nn.Linear(512, 256),
                    nn.ReLU(inplace=True),
                    nn.Dropout(0.2),
                    nn.Linear(256, n_classes)
                )
            else:
                self.heads[attr_name] = nn.Sequential(
                    nn.Linear(512, 128),
                    nn.ReLU(inplace=True),
                    nn.Linear(128, n_classes)
                )

    def forward(self, x):
        backbone_features = self.backbone(x)
        shared = self.shared_fc(backbone_features)
        outputs = {}
        for attr_name, head in self.heads.items():
            outputs[attr_name] = head(shared)
        outputs['features'] = shared
        outputs['backbone_features'] = backbone_features
        return outputs

# ============ 5. LOAD TRAINED WEIGHTS ============
model = DressAttributeExtractor(num_classes).to(device)

checkpoint = torch.load(
    os.path.join(MODELS_DIR, 'dress_attribute_extractor_best.pth'),
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

best_epoch = checkpoint['epoch']
best_val_avg_acc = checkpoint['val_avg_accuracy']

print(f"\n✅ Model restored from Epoch {best_epoch}")
print(f"   Val Avg Accuracy: {best_val_avg_acc:.1f}%")

# ============ 6. LOAD BALANCED DATA ============
balanced_dress_df = pd.read_csv(os.path.join(DATA_DIR, 'dresses_balanced.csv'))

# Recreate train/val/test split (same random state = same split)
from sklearn.model_selection import train_test_split

# Re-encode (needed for split)
for attr_name in encoders:
    col_map = {
        'color': 'mapped_color', 'neckline': 'neckline',
        'dress_length': 'dress_length', 'fabric': 'fabric',
        'pattern': 'pattern', 'sleeve_length': 'sleeve_length',
        'usage': 'mapped_usage', 'season': 'season', 'gender': 'gender'
    }
    col = col_map[attr_name]
    balanced_dress_df[f'{attr_name}_encoded'] = encoders[attr_name].transform(balanced_dress_df[col])

train_df, temp_df = train_test_split(
    balanced_dress_df, test_size=0.2,
    stratify=balanced_dress_df['neckline_encoded'],
    random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5,
    stratify=temp_df['neckline_encoded'],
    random_state=42
)

print(f"\n✅ Data restored:")
print(f"   Total: {len(balanced_dress_df)}")
print(f"   Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ============ 7. RESTORE TRAINING CONFIG ============
BATCH_SIZE = 32
NUM_EPOCHS_STAGE1 = 10
NUM_EPOCHS_STAGE2 = 15
NUM_EPOCHS_TOTAL = NUM_EPOCHS_STAGE1 + NUM_EPOCHS_STAGE2
PATIENCE = 7
UNFREEZE_EPOCH = NUM_EPOCHS_STAGE1

LOSS_WEIGHTS = {
    'color': 2.0, 'neckline': 1.5, 'dress_length': 1.2,
    'fabric': 1.0, 'pattern': 1.0, 'sleeve_length': 1.0,
    'usage': 1.5, 'season': 0.8, 'gender': 0.8,
}

# Note: history is lost, create empty one
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': {k: [] for k in LOSS_WEIGHTS.keys()},
    'val_acc': {k: [] for k in LOSS_WEIGHTS.keys()},
    'lr': []
}

print(f"\n{'='*60}")
print(f"✅ FULL RECOVERY COMPLETE!")
print(f"{'='*60}")
print(f"   Model: DressAttributeExtractor_ResNet152")
print(f"   Best Epoch: {best_epoch}")
print(f"   Best Val Avg Acc: {best_val_avg_acc:.1f}%")
print(f"   All encoders, data, config restored")
print(f"\n🚀 Now run Cell 17 to save inference model!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu

✅ Encoders restored:
   color: 24 classes
   neckline: 15 classes
   dress_length: 5 classes
   fabric: 11 classes
   pattern: 9 classes
   sleeve_length: 6 classes
   usage: 5 classes
   season: 4 classes
   gender: 3 classes

✅ Model restored from Epoch 21
   Val Avg Accuracy: 67.9%

✅ Data restored:
   Total: 13095
   Train: 10476 | Val: 1309 | Test: 1310

✅ FULL RECOVERY COMPLETE!
   Model: DressAttributeExtractor_ResNet152
   Best Epoch: 21
   Best Val Avg Acc: 67.9%
   All encoders, data, config restored

🚀 Now run Cell 17 to save inference model!


CELL 17: Save Final Model & All Artifacts

In [6]:
# ============================================
# CELL 17: Save Final Model & All Artifacts
# ============================================

import json
import pickle

# Load best model
checkpoint = torch.load(os.path.join(MODELS_DIR, 'dress_attribute_extractor_best.pth'), weights_only=False, map_location=torch.device('cpu'))
model.load_state_dict(checkpoint['model_state_dict'])


# ============ SAVE INFERENCE MODEL (for backend) ============
inference_save = {
    'model_state_dict': model.state_dict(),
    'num_classes': num_classes,
    'encoder_data': encoder_data,
    'input_size': [3, 224, 224],
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
    'attributes': list(num_classes.keys()),
}

torch.save(inference_save, os.path.join(MODELS_DIR, 'dress_attribute_extractor_inference.pth'))

# ============ SAVE MODEL INFO ============
model_info = {
    'model_name': 'DressAttributeExtractor_ResNet152',
    'architecture': 'ResNet-152 Multi-Head (2-Stage Fine-tuned)',
    'num_heads': 9,
    'attributes': {
        'color': {'num_classes': num_classes['color'], 'classes': encoder_data['color']['classes']},
        'neckline': {'num_classes': num_classes['neckline'], 'classes': encoder_data['neckline']['classes']},
        'dress_length': {'num_classes': num_classes['dress_length'], 'classes': encoder_data['dress_length']['classes']},
        'fabric': {'num_classes': num_classes['fabric'], 'classes': encoder_data['fabric']['classes']},
        'pattern': {'num_classes': num_classes['pattern'], 'classes': encoder_data['pattern']['classes']},
        'sleeve_length': {'num_classes': num_classes['sleeve_length'], 'classes': encoder_data['sleeve_length']['classes']},
        'usage': {'num_classes': num_classes['usage'], 'classes': encoder_data['usage']['classes']},
        'season': {'num_classes': num_classes['season'], 'classes': encoder_data['season']['classes']},
        'gender': {'num_classes': num_classes['gender'], 'classes': encoder_data['gender']['classes']},
    },
    'total_output_neurons': sum(num_classes.values()),
    'best_epoch': best_epoch,
    'best_val_avg_accuracy': round(best_val_avg_acc, 2),
    'training_config': {
        'stage1_epochs': NUM_EPOCHS_STAGE1,
        'stage2_epochs': NUM_EPOCHS_STAGE2,
        'batch_size': BATCH_SIZE,
        'early_stopping_patience': PATIENCE,
    },
    'dataset': {
        'name': 'Kaggle Fashion Product Images (Apparel)',
        'total_samples': len(balanced_dress_df),
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
    },
    'loss_weights': LOSS_WEIGHTS,
    'pseudo_label_note': 'Neckline, fabric, pattern, sleeve_length, dress_length use pseudo-labels based on articleType heuristics + productDisplayName text mining. Color, usage, season, gender from dataset.',
}

with open(os.path.join(MODELS_DIR, 'model2_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

# Save training history
with open(os.path.join(MODELS_DIR, 'model2_history.pkl'), 'wb') as f:
    pickle.dump(history, f)

# ============ LIST ALL FILES ============
print("=" * 60)
print("📁 ALL SAVED FILES")
print("=" * 60)

for folder_name, folder_path in [('trained_models', MODELS_DIR), ('data', DATA_DIR), ('plots', PLOTS_DIR)]:
    print(f"\n📂 {folder_path}/")
    for f_name in sorted(os.listdir(folder_path)):
        size = os.path.getsize(os.path.join(folder_path, f_name))
        if size > 1024 * 1024:
            print(f"   📄 {f_name} ({size / (1024*1024):.1f} MB)")
        else:
            print(f"   📄 {f_name} ({size / 1024:.1f} KB)")

📁 ALL SAVED FILES

📂 /content/drive/MyDrive/AccessoriesAI/trained_models/
   📄 accessory_classifier_resnet152_best.pth (702.2 MB)
   📄 accessory_classifier_resnet152_inference.pth (234.5 MB)
   📄 accessory_label_encoders.json (0.9 KB)
   📄 dress_attribute_extractor_best.pth (706.0 MB)
   📄 dress_attribute_extractor_inference.pth (235.8 MB)
   📄 dress_label_encoders.json (1.9 KB)
   📄 model1_history.pkl (2.2 KB)
   📄 model1_info.json (1.8 KB)
   📄 model2_history.pkl (0.2 KB)
   📄 model2_info.json (3.1 KB)

📂 /content/drive/MyDrive/AccessoriesAI/data/
   📄 accessories_balanced.csv (1.2 MB)
   📄 accessories_raw_clean.csv (979.4 KB)
   📄 dresses_balanced.csv (2.4 MB)

📂 /content/drive/MyDrive/AccessoriesAI/plots/
   📄 model1_confusion_matrix.png (244.9 KB)
   📄 model1_sample_predictions.png (186.2 KB)
   📄 model1_training_history.png (301.5 KB)
   📄 model2_confusion_matrices.png (641.8 KB)
   📄 model2_sample_predictions.png (225.7 KB)
   📄 model2_training_history.png (399.9 KB)


Cell 18: Final Verification — Backend-Ready Prediction


In [15]:
# ============================================
# CELL 18: Final Verification — Backend-Ready Function
# ============================================

import random  # ← ADD THIS LINE
from PIL import Image

def predict_dress_from_image(image_path):
    """
    Backend-ready function: Takes dress image path → returns all attributes.
    This is exactly how Flask backend will use it.
    """
    checkpoint = torch.load(
        os.path.join(MODELS_DIR, 'dress_attribute_extractor_inference.pth'),
        map_location=device
    )

    # Rebuild model
    inference_model = DressAttributeExtractor(checkpoint['num_classes']).to(device)
    inference_model.load_state_dict(checkpoint['model_state_dict'])
    inference_model.eval()

    # Transform
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=checkpoint['normalize_mean'], std=checkpoint['normalize_std'])
    ])

    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = inference_model(input_tensor)

    result = {}
    for attr_name, attr_info in checkpoint['encoder_data'].items():
        classes = attr_info['classes']
        probs = torch.softmax(outputs[attr_name], dim=1)
        conf, pred_idx = torch.max(probs, 1)

        # Top 3 predictions
        top3_probs, top3_indices = torch.topk(probs, min(3, len(classes)), dim=1)
        top3 = [
            {'label': classes[idx.item()], 'confidence': round(prob.item() * 100, 2)}
            for prob, idx in zip(top3_probs[0], top3_indices[0])
        ]

        result[attr_name] = {
            'prediction': classes[pred_idx.item()],
            'confidence': round(conf.item() * 100, 2),
            'top3': top3
        }

    # Also extract feature vector for Model 3
    features = outputs['features'].cpu().numpy().tolist()[0]  # 512-dim
    result['feature_vector_dim'] = len(features)

    return result

# ============ TEST ============
test_sample = test_df.iloc[random.randint(0, len(test_df)-1)]
print(f"🖼️ Testing: {test_sample['productDisplayName']}")
print(f"   Article Type: {test_sample['articleType']}")

result = predict_dress_from_image(test_sample['image_path'])

print(f"\n{'='*60}")
print(f"🔮 PREDICTION RESULTS (Backend Format)")
print(f"{'='*60}")

for attr, data in result.items():
    if attr == 'feature_vector_dim':
        print(f"\n   📐 Feature Vector: {data}-dimensional")
        continue

    emoji_map = {
        'color': '🎨', 'neckline': '👔', 'dress_length': '📏',
        'fabric': '🧵', 'pattern': '🎭', 'sleeve_length': '💪',
        'usage': '🎯', 'season': '📅', 'gender': '👤'
    }
    emoji = emoji_map.get(attr, '📋')
    print(f"\n   {emoji} {attr.upper()}:")
    print(f"   → {data['prediction']} ({data['confidence']}%)")
    print(f"   Top 3: ", end="")
    for t in data['top3']:
        print(f"{t['label']}({t['confidence']}%) ", end="")
    print()

print(f"\n{'='*60}")
print(f"✅ MODEL 2 TRAINING 100% COMPLETE!")
print(f"{'='*60}")
print(f"\n📋 Summary:")
print(f"   Model: DressAttributeExtractor_ResNet152")
print(f"   Attributes: 9 (Color, Neckline, Length, Fabric, Pattern, Sleeve, Usage, Season, Gender)")
print(f"   Total Output Neurons: {sum(num_classes.values())}")
print(f"   Best Val Avg Accuracy: {best_val_avg_acc:.1f}%")
print(f"   Feature Vector: 512-dim (for Model 3)")
print(f"   Saved To: {MODELS_DIR}")
print(f"\n🚀 Ready for Model 3: Multimodal Fusion Transformer!")

🖼️ Testing: Probase Men Funky Print White Tshirts
   Article Type: Tshirts

🔮 PREDICTION RESULTS (Backend Format)

   🎨 COLOR:
   → White (35.54%)
   Top 3: White(35.54%) Pink(30.85%) Multi-color(11.89%) 

   👔 NECKLINE:
   → Scoop Neck (36.15%)
   Top 3: Scoop Neck(36.15%) Crew Neck(32.36%) V-Neck(26.68%) 

   📏 DRESS_LENGTH:
   → Mini (99.52%)
   Top 3: Mini(99.52%) Knee-length(0.34%) Midi(0.12%) 

   🧵 FABRIC:
   → Cotton (56.92%)
   Top 3: Cotton(56.92%) Polyester(39.85%) Chiffon(0.97%) 

   🎭 PATTERN:
   → Solid (67.47%)
   Top 3: Solid(67.47%) Abstract(29.39%) Striped(1.46%) 

   💪 SLEEVE_LENGTH:
   → Short Sleeve (60.78%)
   Top 3: Short Sleeve(60.78%) Cap Sleeve(36.38%) 3/4 Sleeve(1.12%) 

   🎯 USAGE:
   → Casual (98.95%)
   Top 3: Casual(98.95%) Sports(0.97%) Festive/Religious(0.08%) 

   📅 SEASON:
   → Fall (53.62%)
   Top 3: Fall(53.62%) Summer(45.64%) Winter(0.5%) 

   👤 GENDER:
   → Men (98.76%)
   Top 3: Men(98.76%) Women(1.24%) Unisex(0.0%) 

   📐 Feature Vector: 512-dim

In [13]:
# ============================================
# CELL 18: Final Verification — Backend-Ready Function
# ============================================

import random
from PIL import Image

def predict_dress_from_image(image_path):
    """
    Backend-ready function: Takes dress image path → returns all attributes.
    """
    checkpoint = torch.load(
        os.path.join(MODELS_DIR, 'dress_attribute_extractor_inference.pth'),
        map_location=device
    )

    inference_model = DressAttributeExtractor(checkpoint['num_classes']).to(device)
    inference_model.load_state_dict(checkpoint['model_state_dict'])
    inference_model.eval()

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=checkpoint['normalize_mean'], std=checkpoint['normalize_std'])
    ])

    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = inference_model(input_tensor)

    result = {}
    for attr_name, attr_info in checkpoint['encoder_data'].items():
        classes = attr_info['classes']
        probs = torch.softmax(outputs[attr_name], dim=1)
        conf, pred_idx = torch.max(probs, 1)

        top3_probs, top3_indices = torch.topk(probs, min(3, len(classes)), dim=1)
        top3 = [
            {'label': classes[idx.item()], 'confidence': round(prob.item() * 100, 2)}
            for prob, idx in zip(top3_probs[0], top3_indices[0])
        ]

        result[attr_name] = {
            'prediction': classes[pred_idx.item()],
            'confidence': round(conf.item() * 100, 2),
            'top3': top3
        }

    features = outputs['features'].cpu().numpy().tolist()[0]
    result['feature_vector_dim'] = len(features)

    return result

# ============ TEST ============
# Find a valid test image that exists on disk
test_image_found = False
for i in range(len(test_df)):
    test_sample = test_df.iloc[i]
    if os.path.exists(test_sample['image_path']):
        test_image_found = True
        break

if not test_image_found:
    # If no test images exist (runtime restarted), check if dataset needs re-extraction
    print("⚠️ No test images found! Dataset may need re-extraction.")
    print("   Re-run Cell 2 to download/extract the dataset again.")
    print("   Then re-run this cell.")
else:
    print(f"🖼️ Testing: {test_sample['productDisplayName']}")
    print(f"   Article Type: {test_sample['articleType']}")

    result = predict_dress_from_image(test_sample['image_path'])

    print(f"\n{'='*60}")
    print(f"🔮 PREDICTION RESULTS (Backend Format)")
    print(f"{'='*60}")

    for attr, data in result.items():
        if attr == 'feature_vector_dim':
            print(f"\n   📐 Feature Vector: {data}-dimensional")
            continue

        emoji_map = {
            'color': '🎨', 'neckline': '👔', 'dress_length': '📏',
            'fabric': '🧵', 'pattern': '🎭', 'sleeve_length': '💪',
            'usage': '🎯', 'season': '📅', 'gender': '👤'
        }
        emoji = emoji_map.get(attr, '📋')
        print(f"\n   {emoji} {attr.upper()}:")
        print(f"   → {data['prediction']} ({data['confidence']}%)")
        print(f"   Top 3: ", end="")
        for t in data['top3']:
            print(f"{t['label']}({t['confidence']}%) ", end="")
        print()

    print(f"\n{'='*60}")
    print(f"✅ MODEL 2 TRAINING 100% COMPLETE!")
    print(f"{'='*60}")
    print(f"\n📋 Summary:")
    print(f"   Model: DressAttributeExtractor_ResNet152")
    print(f"   Attributes: 9 (Color, Neckline, Length, Fabric, Pattern, Sleeve, Usage, Season, Gender)")
    print(f"   Total Output Neurons: {sum(num_classes.values())}")
    print(f"   Best Val Avg Accuracy: {best_val_avg_acc:.1f}%")
    print(f"   Feature Vector: 512-dim (for Model 3)")
    print(f"   Saved To: {MODELS_DIR}")
    print(f"\n🚀 Ready for Model 3: Multimodal Fusion Transformer!")

⚠️ No test images found! Dataset may need re-extraction.
   Re-run Cell 2 to download/extract the dataset again.
   Then re-run this cell.


In [14]:
# ============================================
# CELL 2: Download & Extract Kaggle Dataset (FIXED)
# ============================================

from google.colab import files
import os

# Setup Kaggle API
os.makedirs('/root/.kaggle', exist_ok=True)

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("📤 Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json
else:
    print("✅ kaggle.json already exists")

# ============ CLEAN UP OLD FAILED DOWNLOADS ============
DATASET_DIR = '/content/fashion_data'

# Remove old incomplete download
if os.path.exists(DATASET_DIR):
    print("🗑️ Removing old incomplete download...")
    !rm -rf {DATASET_DIR}

os.makedirs(DATASET_DIR, exist_ok=True)

# ============ DOWNLOAD - USE SMALLER VERSION (faster) ============
# The "small" version has same CSV + resized images (saves time)
print("\n📥 Downloading dataset from Kaggle (this may take 5-10 minutes)...")
print("   ⚠️ DO NOT interrupt! Let it complete fully.\n")

!kaggle datasets download -d paramaggarwal/fashion-product-images-small -p {DATASET_DIR} --force

# ============ CHECK DOWNLOAD ============
print("\n🔍 Checking downloaded files...")
!ls -la {DATASET_DIR}/

# Find the zip file
zip_file = None
for f in os.listdir(DATASET_DIR):
    if f.endswith('.zip'):
        zip_file = os.path.join(DATASET_DIR, f)
        size_mb = os.path.getsize(zip_file) / (1024 * 1024)
        print(f"✅ Found zip: {f} ({size_mb:.1f} MB)")
        break

if zip_file is None:
    print("❌ No zip file found! Try downloading again.")
    print("   Or download manually from: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small")
else:
    # ============ EXTRACT ============
    print(f"\n📦 Extracting {zip_file}...")
    !unzip -q -o {zip_file} -d {DATASET_DIR}/
    print("✅ Extraction complete!")

    # ============ FIND FILES ============
    print("\n🔍 Searching for image directory and styles.csv...")

    # Find images
    IMAGE_DIR = None
    for root, dirs, dir_files in os.walk(DATASET_DIR):
        # Look for a directory with .jpg files
        jpg_count = sum(1 for f in dir_files if f.endswith('.jpg'))
        if jpg_count > 100:
            IMAGE_DIR = root
            break
        if 'images' in dirs:
            candidate = os.path.join(root, 'images')
            if os.path.isdir(candidate):
                IMAGE_DIR = candidate
                break

    # Find styles.csv
    STYLES_CSV = None
    for root, dirs, dir_files in os.walk(DATASET_DIR):
        for f in dir_files:
            if f == 'styles.csv':
                STYLES_CSV = os.path.join(root, f)
                break
        if STYLES_CSV:
            break

    # ============ VERIFY ============
    if IMAGE_DIR:
        total_images = len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')])
        print(f"\n✅ Image Directory: {IMAGE_DIR}")
        print(f"   Total images: {total_images}")
        print(f"   Sample: {[f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')][:5]}")
    else:
        print("\n❌ Could not find images! Showing folder structure:")
        !find {DATASET_DIR} -type d | head -20

    if STYLES_CSV:
        print(f"\n✅ Styles CSV: {STYLES_CSV}")
    else:
        print("\n❌ Could not find styles.csv!")
        !find {DATASET_DIR} -name "*.csv" | head -10

📤 Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json

📥 Downloading dataset from Kaggle (this may take 5-10 minutes)...
   ⚠️ DO NOT interrupt! Let it complete fully.

Dataset URL: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small
License(s): MIT
 97% 546M/565M [00:06<00:00, 196MB/s]
100% 565M/565M [00:06<00:00, 95.6MB/s]

🔍 Checking downloaded files...
total 578740
drwxr-xr-x 2 root root      4096 Mar  4 14:32 .
drwxr-xr-x 1 root root      4096 Mar  4 14:32 ..
-rw-r--r-- 1 root root 592614936 Oct 22  2019 fashion-product-images-small.zip
✅ Found zip: fashion-product-images-small.zip (565.2 MB)

📦 Extracting /content/fashion_data/fashion-product-images-small.zip...
✅ Extraction complete!

🔍 Searching for image directory and styles.csv...

✅ Image Directory: /content/fashion_data/images
   Total images: 44441
   Sample: ['13425.jpg', '22177.jpg', '45129.jpg', '59678.jpg', '2132.jpg']

✅ Styles CSV: /content/fashion_data/styles.csv
